In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
def visualize_span(v1, v2, test_point=None, axis_limit=5):
    """
    Визуализирует охват (Span) двух векторов и опционально тестовую точку.

    v1, v2: векторы (list или np.array)
    test_point: точка для проверки (list или np.array или None)
    axis_limit: масштаб отображения
    """
    # Гарантируем, что работаем с массивами numpy для математических операций
    v1 = np.array(v1, dtype=float)
    v2 = np.array(v2, dtype=float)

    # 1. Создаем плоскость подпространства
    scalars = np.linspace(-axis_limit, axis_limit, 30)
    s1, s2 = np.meshgrid(scalars, scalars)
    X = s1 * v1[0] + s2 * v2[0]
    Y = s1 * v1[1] + s2 * v2[1]
    Z = s1 * v1[2] + s2 * v2[2]

    fig = go.Figure()

    # 2. Отрисовка подпространства (плоскости)
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        opacity=0.4,
        colorscale='Blues',
        showscale=False,
        name="Подпространство (Span)",
        showlegend=True
    ))

    # 3. Отрисовка базовых векторов
    for v, color, name in zip([v1, v2], ['red', 'green'], ['v1', 'v2']):
        fig.add_trace(go.Scatter3d(
            x=[0, v[0]], y=[0, v[1]], z=[0, v[2]],
            mode='lines+markers',
            line=dict(color=color, width=6),
            marker=dict(size=4),
            name=f"Вектор {name} {v.tolist()}"
        ))

    # 4. Начало координат
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode='markers',
        marker=dict(size=5, color='black'), name="Центр (0,0,0)"
    ))

    # 5. Если передана тестовая точка — проверяем её и рисуем
    if test_point is not None:
        tp = np.array(test_point, dtype=float)

        # Математическая проверка на принадлежность плоскости (через ранги)
        basis_matrix = np.column_stack((v1, v2))
        rank_basis = np.linalg.matrix_rank(basis_matrix)
        rank_augmented = np.linalg.matrix_rank(np.column_stack((basis_matrix, tp)))

        on_plane = (rank_basis == rank_augmented)
        color = 'purple' if on_plane else 'orange'
        status = "на плоскости" if on_plane else "вне плоскости"

        fig.add_trace(go.Scatter3d(
            x=[tp[0]], y=[tp[1]], z=[tp[2]],
            mode='markers+text',
            marker=dict(size=10, color=color, symbol='diamond'),
            text=[f"Точка {status}"],
            textposition="top center",
            name=f"Тест: {tp.tolist()}"
        ))

    # 6. Настройка "классического" вида осей
    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-axis_limit, axis_limit], title='X'),
            yaxis=dict(range=[-axis_limit, axis_limit], title='Y'),
            zaxis=dict(range=[-axis_limit, axis_limit], title='Z'),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        legend=dict(x=0, y=1)
    )

    fig.show()

In [3]:
v1 = [1, 0, 2]
v2 = [-1, 1, 2]

In [4]:
# Пример 1: Только векторы и плоскость
visualize_span(v1, v2)

In [8]:
# Пример 2: С точкой, которая лежит на плоскости (сумма v1 + v2)
tp_on = np.array(v1) + np.array(v2)
visualize_span(v1, v2, test_point=tp_on)

In [10]:
# Пример 3: С точкой, которая НЕ лежит на плоскости
visualize_span(v1, v2, test_point=[0, 0, 5])